[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laverde97/phd-data-science-ai/blob/main/semesters/semester-01/machine-learning/notes/notebooks/02-random-forest-regresion.ipynb)

# Machine Learning: Random Forest para Regresión

## Colab completo con datos reales, métricas, gráficos, optimización y predicciones

### Objetivos
Al terminar este notebook podrás:
- comprender qué es Random Forest y cómo se relaciona con un Árbol de Decisión;
- entender **bagging, bootstrap, aleatoriedad de variables y OOB Score**;
- entrenar un `RandomForestRegressor`;
- evaluar con **MAE, MedAE, MSE, RMSE, R² y MAPE**;
- analizar Train vs Test y detectar sobreajuste;
- estudiar `n_estimators` y `max_depth`;
- optimizar hiperparámetros con `GridSearchCV`;
- realizar predicciones concretas y medir sus errores;
- analizar Real vs Predicho, residuos e importancia de variables.

> El dataset tiene contexto médico, pero este ejercicio es exclusivamente académico y no debe utilizarse para decisiones clínicas.


# 1. ¿Qué es un problema de regresión?

La **regresión** se utiliza cuando la variable objetivo `y` es numérica.

Ejemplos reales:

| Problema | X | y |
|---|---|---|
| Vivienda | área, habitaciones, ubicación | precio |
| Ventas | publicidad, precio, temporada | ventas |
| Energía | clima, hora, ocupación | consumo |
| Logística | distancia, tráfico, vehículo | tiempo de entrega |

En este notebook utilizaremos Random Forest para predecir una variable cuantitativa.


# 2. Del Árbol de Decisión a Random Forest

Un Árbol de Decisión utiliza reglas sucesivas para llegar a una predicción.

Su ventaja es la interpretabilidad, pero puede ser inestable y sobreajustarse.

**Random Forest** mejora la estabilidad entrenando muchos árboles diferentes y combinando sus resultados.


# 3. ¿Qué es Random Forest?

Random Forest es un método de **ensamble**.

En regresión:

1. crea muchas muestras del conjunto Train;
2. entrena un árbol sobre cada muestra;
3. introduce aleatoriedad en las variables disponibles para dividir nodos;
4. cada árbol genera una predicción;
5. la predicción final es el **promedio**.

Si cinco árboles predicen 150, 160, 145, 155 y 140:

\[
\hat y = \frac{150+160+145+155+140}{5}=150
\]

La combinación de árboles suele ser más estable que depender de uno solo.


# 4. Conceptos fundamentales

## Bagging
Entrenar muchos modelos sobre muestras diferentes y combinar sus resultados.

## Bootstrap
Cada árbol recibe una muestra aleatoria **con reemplazo** del Train.

## Aleatoriedad de variables
En cada división, el árbol considera solo un subconjunto de variables.

## Promedio
En regresión, Random Forest promedia las predicciones de todos los árboles.

## OOB — Out-of-Bag
Las observaciones que no fueron seleccionadas para un árbol pueden utilizarse para evaluar ese árbol. Al combinar esas predicciones obtenemos una estimación adicional del rendimiento.


# 5. ¿Cuándo utilizar Random Forest?

### Puede ser útil cuando:
- existen relaciones no lineales;
- hay interacciones entre variables;
- trabajamos con datos tabulares;
- buscamos un modelo robusto como benchmark;
- un único árbol resulta demasiado inestable.

### Ventajas
- buen desempeño en muchos problemas tabulares;
- poca necesidad de transformaciones;
- no requiere escalamiento obligatorio;
- captura no linealidades e interacciones;
- permite importancia de variables;
- dispone de OOB Score.

### Limitaciones
- menos interpretable que un solo árbol;
- más costoso computacionalmente;
- la importancia tradicional no implica causalidad;
- no necesariamente mejora indefinidamente al agregar más árboles.


# 6. Dataset real: Diabetes de scikit-learn

Usaremos el mismo dataset del ejercicio de Árbol de Decisión para facilitar la comparación.

- **442 observaciones**
- **10 variables predictoras**
- objetivo numérico

Variables: `age`, `sex`, `bmi`, `bp`, `s1`, `s2`, `s3`, `s4`, `s5`, `s6`.


In [ ]:
# Librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# Cargar datos
diabetes = load_diabetes(as_frame=True, scaled=False)

X = diabetes.data.copy()
y = diabetes.target.copy()

df = X.copy()
df["target"] = y

print("Dimensiones:", df.shape)
display(df.head())


# 7. Exploración inicial — EDA

In [ ]:
print("Tipos de variables:")
display(df.dtypes.to_frame("dtype"))

print("\nValores faltantes:")
display(df.isna().sum().to_frame("faltantes"))

print("\nResumen estadístico:")
display(df.describe().T)


In [ ]:
# Gráfico 1: distribución del objetivo
plt.figure(figsize=(8,5))
plt.hist(y, bins=25)
plt.title("Distribución de la variable objetivo")
plt.xlabel("Progresión cuantitativa")
plt.ylabel("Frecuencia")
plt.show()


In [ ]:
# Gráfico 2: correlaciones
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10,8))
img = plt.imshow(corr.values, aspect="auto")
plt.colorbar(img, label="Correlación")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Matriz de correlaciones")
plt.tight_layout()
plt.show()


# 8. Separación Train/Test

- **Train:** aprende el modelo.
- **Test:** evalúa generalización.

Usaremos 80% / 20%.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


# 9. Modelo Random Forest baseline

Usaremos inicialmente:

- `n_estimators=200`
- `max_depth=5`
- `min_samples_leaf=3`
- `bootstrap=True`
- `oob_score=True`

`n_estimators` es el número de árboles del bosque.


In [ ]:
modelo_baseline = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=3,
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

modelo_baseline.fit(X_train, y_train)

pred_train_baseline = modelo_baseline.predict(X_train)
pred_test_baseline = modelo_baseline.predict(X_test)

print("OOB Score:", round(modelo_baseline.oob_score_, 4))


# 10. ¿Qué es OOB Score?

Con bootstrap, algunas observaciones quedan fuera de la muestra de entrenamiento de cada árbol.

Esas observaciones pueden usarse para generar predicciones **Out-of-Bag**.

En regresión, `oob_score_` se expresa por defecto como **R²**.

- más cerca de 1 → mejor;
- alrededor de 0 → similar a predecir la media;
- negativo → rendimiento pobre.

OOB es una evaluación adicional; no sustituye completamente al conjunto Test.


# 11. Métricas de regresión

### MAE
Error absoluto promedio. **Menor = mejor.**

### MedAE
Mediana del error absoluto. Robusta frente a errores extremos.

### MSE
Promedio del error al cuadrado. Penaliza errores grandes.

### RMSE
Raíz del MSE. Está en las mismas unidades de `y`. **Menor = mejor.**

### R²
Compara el modelo con predecir siempre la media. **Mayor = mejor.**

### MAPE
Error porcentual promedio. Debe interpretarse con precaución cuando `y` contiene valores cercanos a cero.


In [ ]:
def metricas_regresion(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MedAE": median_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "MAPE_%": mean_absolute_percentage_error(y_true, y_pred) * 100
    }

metricas_train_baseline = metricas_regresion(y_train, pred_train_baseline)
metricas_test_baseline = metricas_regresion(y_test, pred_test_baseline)

display(pd.DataFrame({
    "Train": metricas_train_baseline,
    "Test": metricas_test_baseline
}).T)


# 12. Underfitting y Overfitting

### Underfitting
El bosque es demasiado simple:
- profundidad muy baja;
- hojas demasiado grandes;
- pocas variables disponibles.

### Overfitting
Random Forest suele sobreajustar menos que un árbol individual, pero puede ocurrir.

La señal principal sigue siendo una diferencia grande entre Train y Test.


# 13. ¿Cuántos árboles necesito?

In [ ]:
cantidades_arboles = [10, 25, 50, 100, 200, 300]

rmse_train_n = []
rmse_test_n = []

for n in cantidades_arboles:
    modelo = RandomForestRegressor(
        n_estimators=n,
        max_depth=5,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    modelo.fit(X_train, y_train)

    rmse_train_n.append(root_mean_squared_error(y_train, modelo.predict(X_train)))
    rmse_test_n.append(root_mean_squared_error(y_test, modelo.predict(X_test)))

plt.figure(figsize=(9,5))
plt.plot(cantidades_arboles, rmse_train_n, marker="o", label="Train")
plt.plot(cantidades_arboles, rmse_test_n, marker="o", label="Test")
plt.title("Número de árboles vs RMSE")
plt.xlabel("n_estimators")
plt.ylabel("RMSE")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## Interpretación

Al aumentar `n_estimators`, el bosque suele estabilizarse.

No esperamos que el error baje indefinidamente. Llega un momento en el que agregar árboles aporta muy poca mejora y solo aumenta el tiempo de cálculo.


# 14. Profundidad máxima vs desempeño

In [ ]:
profundidades = [2, 3, 4, 5, 6, 8, None]

rmse_train_depth = []
rmse_test_depth = []

for depth in profundidades:
    modelo = RandomForestRegressor(
        n_estimators=150,
        max_depth=depth,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    modelo.fit(X_train, y_train)

    rmse_train_depth.append(root_mean_squared_error(y_train, modelo.predict(X_train)))
    rmse_test_depth.append(root_mean_squared_error(y_test, modelo.predict(X_test)))

labels = [str(x) if x is not None else "None" for x in profundidades]
pos = np.arange(len(labels))

plt.figure(figsize=(9,5))
plt.plot(pos, rmse_train_depth, marker="o", label="Train")
plt.plot(pos, rmse_test_depth, marker="o", label="Test")
plt.xticks(pos, labels)
plt.title("max_depth vs RMSE")
plt.xlabel("max_depth")
plt.ylabel("RMSE")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# 15. Hiperparámetros importantes

- **`n_estimators`**: número de árboles.
- **`max_depth`**: profundidad máxima.
- **`min_samples_split`**: mínimo de observaciones para dividir un nodo.
- **`min_samples_leaf`**: mínimo de observaciones en una hoja.
- **`max_features`**: variables consideradas en cada división.
- **`bootstrap`**: activa muestras con reemplazo.
- **`max_samples`**: tamaño de cada muestra bootstrap.
- **`criterion`**: función para evaluar divisiones.
- **`random_state`**: reproducibilidad.


# 16. Validación cruzada

In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

resultado_cv = cross_validate(
    modelo_baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

resumen_cv = pd.DataFrame({
    "MAE": -resultado_cv["test_MAE"],
    "RMSE": -resultado_cv["test_RMSE"],
    "R2": resultado_cv["test_R2"]
})

display(resumen_cv)
display(resumen_cv.mean().to_frame("Promedio").T)


# 17. Optimización con GridSearchCV

Buscaremos una configuración mejor usando únicamente **Train + validación cruzada**.

La cuadrícula es deliberadamente moderada para que pueda ejecutarse cómodamente en Google Colab.


In [ ]:
modelo_grid = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, None],
    "min_samples_leaf": [1, 4],
    "max_features": ["sqrt", 1.0]
}

grid = GridSearchCV(
    estimator=modelo_grid,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Mejores hiperparámetros:")
print(grid.best_params_)

print("\nMejor RMSE promedio de CV:")
print(-grid.best_score_)


# 18. Evaluación final sobre Test

In [ ]:
mejor_modelo = grid.best_estimator_

pred_train_final = mejor_modelo.predict(X_train)
pred_test_final = mejor_modelo.predict(X_test)

metricas_train_final = metricas_regresion(y_train, pred_train_final)
metricas_test_final = metricas_regresion(y_test, pred_test_final)

display(pd.DataFrame({
    "Train": metricas_train_final,
    "Test": metricas_test_final
}).T)


In [ ]:
# Baseline vs modelo optimizado
display(pd.DataFrame({
    "Baseline Random Forest": metricas_test_baseline,
    "Random Forest optimizado": metricas_test_final
}).T)


# 19. Predicción individual

Tomaremos una observación real de Test y compararemos:

- valor real;
- predicción;
- error;
- error absoluto;
- error porcentual.


In [ ]:
posicion = 0

observacion = X_test.iloc[[posicion]]
valor_real = y_test.iloc[posicion]
prediccion_individual = mejor_modelo.predict(observacion)[0]

error = valor_real - prediccion_individual
error_absoluto = abs(error)
error_porcentual = error_absoluto / abs(valor_real) * 100

print("VALOR REAL       :", round(valor_real, 2))
print("VALOR PREDICHO   :", round(prediccion_individual, 2))
print("ERROR            :", round(error, 2))
print("ERROR ABSOLUTO   :", round(error_absoluto, 2))
print("ERROR PORCENTUAL :", round(error_porcentual, 2), "%")

display(observacion)


Una sola predicción no basta para juzgar el modelo. Debemos analizar muchas observaciones y las métricas globales.


# 20. Predicciones para varias observaciones

In [ ]:
n_ejemplos = min(15, len(X_test))

muestra_test = X_test.sample(n=n_ejemplos, random_state=RANDOM_STATE)
real_muestra = y_test.loc[muestra_test.index]
pred_muestra = mejor_modelo.predict(muestra_test)

tabla_predicciones = pd.DataFrame({
    "Real": real_muestra.to_numpy(),
    "Predicción": pred_muestra
}, index=muestra_test.index)

tabla_predicciones["Error"] = tabla_predicciones["Real"] - tabla_predicciones["Predicción"]
tabla_predicciones["Error absoluto"] = tabla_predicciones["Error"].abs()
tabla_predicciones["Error %"] = (
    tabla_predicciones["Error absoluto"] / tabla_predicciones["Real"].abs() * 100
)

display(tabla_predicciones.round(2))


In [ ]:
# Gráfico: Real vs Predicción para 15 observaciones
tabla_plot = tabla_predicciones.reset_index(drop=True)
x = np.arange(len(tabla_plot))
ancho = 0.38

plt.figure(figsize=(12,6))
plt.bar(x-ancho/2, tabla_plot["Real"], width=ancho, label="Real")
plt.bar(x+ancho/2, tabla_plot["Predicción"], width=ancho, label="Predicción")
plt.title("Valores reales vs predicciones")
plt.xlabel("Observaciones")
plt.ylabel("Valor objetivo")
plt.xticks(x, [f"Obs {i+1}" for i in range(len(tabla_plot))], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


# 21. Predicciones dentro de márgenes de error

En regresión no debemos llamar “accuracy” a este resultado.

Podemos comunicar qué porcentaje de predicciones queda dentro de:

- ±10%
- ±20%
- ±30%

Es una medida complementaria, no un sustituto de MAE, RMSE o R².


In [ ]:
y_test_array = y_test.to_numpy()
pred_test_array = np.asarray(pred_test_final)

ape = np.abs(y_test_array - pred_test_array) / np.abs(y_test_array) * 100

resumen_tolerancias = pd.DataFrame({
    "Margen de error": ["≤ 10%", "≤ 20%", "≤ 30%"],
    "Predicciones dentro del margen (%)": [
        np.mean(ape <= 10) * 100,
        np.mean(ape <= 20) * 100,
        np.mean(ape <= 30) * 100
    ]
})

display(resumen_tolerancias.round(2))


In [ ]:
plt.figure(figsize=(8,5))
plt.bar(
    resumen_tolerancias["Margen de error"],
    resumen_tolerancias["Predicciones dentro del margen (%)"]
)
plt.ylim(0,100)
plt.title("Predicciones dentro de diferentes márgenes")
plt.xlabel("Margen")
plt.ylabel("Predicciones (%)")

for i, valor in enumerate(resumen_tolerancias["Predicciones dentro del margen (%)"]):
    plt.text(i, valor+2, f"{valor:.1f}%", ha="center")

plt.show()


# 22. Gráfico global: Real vs Predicho

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, pred_test_final, alpha=0.7)

minimo = min(y_test.min(), pred_test_final.min())
maximo = max(y_test.max(), pred_test_final.max())

plt.plot([minimo,maximo], [minimo,maximo], linestyle="--")
plt.title("Random Forest: Real vs Predicho")
plt.xlabel("Valor real")
plt.ylabel("Predicción")
plt.grid(alpha=0.2)
plt.show()


Los puntos cercanos a la diagonal representan mejores predicciones. Una gran dispersión indica mayor error.


# 23. Residuos

In [ ]:
residuos = y_test.to_numpy() - pred_test_final

plt.figure(figsize=(8,5))
plt.scatter(pred_test_final, residuos, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.title("Residuos vs predicciones")
plt.xlabel("Predicción")
plt.ylabel("Residuo = real - predicho")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(residuos, bins=20)
plt.axvline(0, linestyle="--")
plt.title("Distribución de residuos")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.show()


Idealmente los residuos deben distribuirse alrededor de cero sin patrones demasiado claros.


# 24. Casos con mayor error

In [ ]:
errores_test = pd.DataFrame({
    "Real": y_test.to_numpy(),
    "Predicción": pred_test_final
})

errores_test["Error absoluto"] = np.abs(
    errores_test["Real"] - errores_test["Predicción"]
)

errores_ordenados = errores_test.sort_values(
    "Error absoluto",
    ascending=False
).reset_index(drop=True)

display(errores_ordenados.head(10).round(2))


In [ ]:
top_errores = errores_ordenados.head(10)

plt.figure(figsize=(10,5))
plt.bar(
    [f"Caso {i+1}" for i in range(len(top_errores))],
    top_errores["Error absoluto"]
)
plt.title("10 observaciones con mayor error absoluto")
plt.xlabel("Observación")
plt.ylabel("Error absoluto")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 25. Importancia de variables

In [ ]:
importancias = pd.Series(
    mejor_modelo.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(importancias.to_frame("Importancia"))


In [ ]:
plt.figure(figsize=(8,5))
plt.bar(importancias.index, importancias.values)
plt.title("Importancia de variables — Random Forest")
plt.xlabel("Variable")
plt.ylabel("Importancia")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


> La importancia indica cuánto utilizó el modelo una variable para reducir error. **No demuestra causalidad**.


# 26. Predicción para una nueva observación

In [ ]:
nueva_observacion = X_train.median().to_frame().T

display(nueva_observacion)

prediccion_nueva = mejor_modelo.predict(nueva_observacion)[0]

print("Predicción del Random Forest:", round(prediccion_nueva, 2))


# 27. Árbol de Decisión vs Random Forest

| Aspecto | Árbol | Random Forest |
|---|---|---|
| Árboles | 1 | Muchos |
| Interpretabilidad | Alta | Media |
| Estabilidad | Menor | Mayor |
| Overfitting | Más probable | Generalmente menor |
| No linealidad | Sí | Sí |
| Predicción | Una hoja | Promedio |
| OOB Score | No | Sí |
| Feature importance | Sí | Sí |

Random Forest sacrifica parte de la interpretabilidad para ganar estabilidad.


# 28. Casos de uso reales

### Vivienda
Predecir precio.

### Ventas
Predecir demanda.

### Energía
Predecir consumo.

### Agricultura
Predecir rendimiento.

### Logística
Predecir tiempos de entrega.

### Salud
Modelar resultados cuantitativos bajo validación científica adecuada.


# 29. ¿Qué métricas recomiendo reportar?

Como mínimo:

**MAE + RMSE + R²**

Para Random Forest también puede ser útil:

**OOB Score**

Y complementar con:

- Train vs Test;
- validación cruzada;
- Real vs Predicho;
- residuos;
- márgenes de error;
- análisis de peores casos.


# 30. Errores comunes

1. Pensar que más árboles siempre significa mucha más precisión.
2. Evaluar únicamente Train.
3. Optimizar mirando Test.
4. Reportar solo R².
5. Llamar “accuracy” al desempeño de regresión.
6. Interpretar importancia como causalidad.
7. Creer que Random Forest nunca sobreajusta.
8. No revisar predicciones concretas ni casos con errores grandes.


# 31. Flujo recomendado

**Problema → EDA → X/y → Train/Test → Baseline → Métricas → OOB → Validación cruzada → Ajuste de hiperparámetros → Test final → Predicciones → Residuos → Importancia → Comparación con otros modelos**


# 32. Conclusiones

Random Forest:

- combina muchos árboles;
- usa bootstrap y aleatoriedad de variables;
- promedia predicciones en regresión;
- suele ser más estable que un árbol individual;
- permite OOB Score;
- requiere evaluar Train, Test y validación cruzada;
- debe interpretarse mediante métricas y gráficos, no con una sola cifra.

La idea central es:

> **No buscamos un bosque que memorice Train, sino uno que generalice bien a observaciones nuevas.**


# 33. Referencias

- Scikit-learn — RandomForestRegressor  
  https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

- Scikit-learn — Forests  
  https://scikit-learn.org/stable/modules/ensemble.html#forest

- Scikit-learn — Diabetes dataset  
  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html

- Scikit-learn — Regression metrics  
  https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics

- Scikit-learn — GridSearchCV  
  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html
